# Story 001 — `VFSResult` unification + query narrowing

Walks through the uncommitted changes from story `001-unify-vfs-result-and-optimize-queries`:

1. **One row shape, one envelope.** `Entry` + `LineMatch` replace `Candidate` + `Detail`. Every op returns the same `VFSResult(function=..., entries=[...])`.
2. **Composition.** `|` / `&` / `-` across any two results; cross-function merges self-label as `hybrid`.
3. **Projection-driven rendering.** `to_str(projection=...)` is the one render entry point — `default` / `all` sentinels included.
4. **CLI `--output` flag.** Widens the SELECT, not just the rendered output.
5. **Narrow SELECTs everywhere.** Vectors + content stay out of default reads; SQL capture proves it.
6. **Hydration via `fs.read`.** When a projected column is missing, the executor issues **one** narrowed `read` call — never a private bypass.

## 0. Setup — in-memory SQLite + a small corpus

In [27]:
from sqlalchemy.ext.asyncio import create_async_engine
from sqlalchemy.pool import StaticPool
from sqlmodel import SQLModel

from vfs.backends.database import DatabaseFileSystem

engine = create_async_engine(
    "sqlite+aiosqlite://",
    poolclass=StaticPool,
    connect_args={"check_same_thread": False},
)
async with engine.begin() as conn:
    await conn.run_sync(SQLModel.metadata.drop_all)
    await conn.run_sync(SQLModel.metadata.create_all)

db = DatabaseFileSystem(engine=engine)
print(type(db).__name__, "ready")

DatabaseFileSystem ready


In [28]:
# Seed a namespace covering files (markdown + python) and a connection edge
await db.write("/docs/intro.md", "# Intro\nhydrate the index\nwelcome\n")
await db.write("/docs/guide.md", "# Guide\nhydrate downstream\n")
await db.write("/src/auth.py", "def login():\n    hydrate(user)\n    return True\n")
await db.write("/src/db.py", "def query():\n    return rows\n")
# A connection edge (imports) — lives under /src/auth.py/.connections/
await db.mkconn(source="/src/auth.py", target="/src/db.py", connection_type="imports")
print("seeded")

seeded


## 1. New `Entry` + `VFSResult` shape

Every op returns a `VFSResult`. `function` identifies the producer; `entries` is a flat list of `Entry`.

In [29]:
result = await db.glob("**/*.md")
print("function:", result.function)
print("entries:", len(result.entries))
for e in result.entries:
    print(" ", e)

function: glob
entries: 2
  path='/docs/guide.md' kind='file' lines=None content=None size_bytes=27 score=None in_degree=None out_degree=None updated_at=datetime.datetime(2026, 4, 19, 20, 35, 45, 270778)
  path='/docs/intro.md' kind='file' lines=None content=None size_bytes=34 score=None in_degree=None out_degree=None updated_at=datetime.datetime(2026, 4, 19, 20, 35, 45, 267720)


In [30]:
# Entry is a flat shape. Inspect the fields:
from vfs.results import Entry

list(Entry.model_fields.keys())

['path',
 'kind',
 'lines',
 'content',
 'size_bytes',
 'score',
 'in_degree',
 'out_degree',
 'updated_at']

In [31]:
# Entry.name is the last path segment (computed property)
e = result.entries[0]
e.path, e.name, e.kind

('/docs/guide.md', 'guide.md', 'file')

## 2. Composition — `|` `&` `-` with `hybrid` relabeling

Same-function merge keeps the `function`. Cross-function merge sets `function="hybrid"`. Left entry wins on overlap.

In [32]:
md = await db.glob("**/*.md")
py = await db.glob("**/*.py")

both = md | py
print("same function merge:", (md | md).function)  # glob
print("cross-function merge:", (md | await db.grep("hydrate")).function)  # hybrid
print("union entries:", sorted(e.path for e in both.entries))
print("difference:", sorted(e.path for e in (both - py).entries))

same function merge: glob
cross-function merge: hybrid
union entries: ['/docs/guide.md', '/docs/intro.md', '/src/auth.py', '/src/db.py']
difference: ['/docs/guide.md', '/docs/intro.md']


In [33]:
# Enrichment chains: .sort / .top / .filter / .kinds — same API on any function's result
grep = await db.grep("hydrate")
print("grep matches:", [e.path for e in grep.entries])
print("top 2 by score:", [e.path for e in (await db.lexical_search("hydrate", k=5)).top(2).entries])
print("only .md:", [e.path for e in md.filter(lambda e: e.path.endswith(".md")).entries])

grep matches: ['/docs/guide.md', '/docs/intro.md', '/src/auth.py']
top 2 by score: ['/docs/guide.md', '/docs/intro.md']
only .md: ['/docs/guide.md', '/docs/intro.md']


## 3. `to_str(projection=...)` — the one render surface

Default projection is per-function. Override with Entry field names or the `default` / `all` sentinels. Order is preserved.

In [34]:
# Default — glob renders a path list
print(md.to_str())

/docs/guide.md
/docs/intro.md


In [35]:
md

VFSResult(success=True, errors=[], function='glob', entries=[Entry(path='/docs/guide.md', kind='file', lines=None, content=None, size_bytes=27, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 19, 20, 35, 45, 270778)), Entry(path='/docs/intro.md', kind='file', lines=None, content=None, size_bytes=34, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 19, 20, 35, 45, 267720))])

In [36]:
# Override: include size_bytes and updated_at
print(md.to_str(projection=("path", "size_bytes", "updated_at")))

| path           | size_bytes | updated_at                 |
| -------------- | ---------: | -------------------------- |
| /docs/guide.md |         27 | 2026-04-19 20:35:45.270778 |
| /docs/intro.md |         34 | 2026-04-19 20:35:45.267720 |


In [37]:
# Sentinel: `default` expands to the function's default projection, then you append columns
stat = await db.stat("/docs/intro.md")
print("--- default ---")
print(stat.to_str())
print()
print("--- all (populated fields only) ---")
print(stat.to_str(projection=("all",)))

--- default ---
/docs/intro.md
  kind: file
  size_bytes: 34
  updated_at: 2026-04-19 20:35:45.267720

--- all (populated fields only) ---
/docs/intro.md
  kind: file
  size_bytes: 34
  updated_at: 2026-04-19 20:35:45.267720


In [38]:
# JSON serialization for APIs / MCP — no `candidates` or `details` keys anywhere
print(stat.to_json())

{"success":true,"errors":[],"function":"stat","entries":[{"path":"/docs/intro.md","kind":"file","size_bytes":34,"updated_at":"2026-04-19T20:35:45.267720"}]}


In [39]:
# grep renders one line per LineMatch, with sliced context when `content` is in the projection
grep_ctx = await db.grep("hydrate", before_context=1, after_context=1, columns=frozenset({'path', 'content', 'size_bytes'}))
print(grep_ctx.to_str(projection=('path', 'size_bytes')))

| path           | size_bytes |
| -------------- | ---------: |
| /docs/guide.md |         27 |
| /docs/intro.md |         34 |
| /src/auth.py   |         47 |


## 4. CLI `--output` — parsed into the plan, threaded to the SELECT

The flag is top-level (may appear anywhere in the query). Unknown fields raise at parse time — before any work happens.

In [40]:
from vfs.query.parser import parse_query, QuerySyntaxError
from vfs.query.executor import execute_query
from vfs.query.render import render_query_result

plan = parse_query('glob "**/*.md" --output path,updated_at,size_bytes')
print("projection on plan:", plan.projection)

result = await execute_query(db, plan)
print(render_query_result(result, plan))

projection on plan: ('path', 'updated_at', 'size_bytes')
| path           | updated_at                 | size_bytes |
| -------------- | -------------------------- | ---------: |
| /docs/guide.md | 2026-04-19 20:35:45.270778 |         27 |
| /docs/intro.md | 2026-04-19 20:35:45.267720 |         34 |


In [41]:
# Unknown field → QuerySyntaxError *before* the pipeline runs
try:
    parse_query("grep hydrate --output bogus")
except QuerySyntaxError as exc:
    print("rejected at parse time:", exc)

rejected at parse time: unknown field 'bogus'


## 5. Query narrowing — no more `select(self._model)` on the hot path

Listen on SQLAlchemy's `before_cursor_execute` and prove that default reads do not project `vfs_objects.embedding` or (for non-text ops) `vfs_objects.content`.

In [42]:
from sqlalchemy import event
import re


class SQLCapture:
    def __init__(self):
        self.statements: list[str] = []
        self._re = re.compile(r"\bFROM\s+vfs_objects\b", re.IGNORECASE)

    def reset(self):
        self.statements.clear()

    def reads(self):
        return [s for s in self.statements if s.lstrip().upper().startswith("SELECT") and self._re.search(s)]

    def mentions(self, col: str) -> bool:
        needle = re.compile(rf"\bvfs_objects\.{re.escape(col)}\b", re.IGNORECASE)
        return any(needle.search(s) for s in self.reads())


capture = SQLCapture()
sync_engine = engine.sync_engine


def _on_execute(_c, _cur, stmt, _params, _ctx, _many):
    capture.statements.append(stmt)


event.listen(sync_engine, "before_cursor_execute", _on_execute)
print("capture armed")

capture armed


In [43]:
# Default glob — should NOT select embedding or content
capture.reset()
await db.glob("**/*.md")
print("SELECTs against vfs_objects:", len(capture.reads()))
print("mentions embedding:", capture.mentions("embedding"))
print("mentions content:  ", capture.mentions("content"))
print("mentions path:     ", capture.mentions("path"))

SELECTs against vfs_objects: 1
mentions embedding: False
mentions content:   False
mentions path:      True


In [44]:
# CLI `--output updated_at` must widen the SELECT to include that column
capture.reset()
await execute_query(db, parse_query('glob "**/*.md" --output default,updated_at'))
print("mentions updated_at:", capture.mentions("updated_at"))
print("mentions embedding :", capture.mentions("embedding"))  # still no
print("mentions content   :", capture.mentions("content"))  # still no

mentions updated_at: True
mentions embedding : False
mentions content   : False


In [45]:
# Compare: read *does* need content — the column shows up only where legitimate
capture.reset()
await db.read("/docs/intro.md")
print("read mentions content:  ", capture.mentions("content"))
print("read mentions embedding:", capture.mentions("embedding"))  # still no

read mentions content:   True
read mentions embedding: False


## 6. Hydration via `fs.read` — one shared surface, one call

When a projected column is null for every entry returned by a stage, the executor backfills via `fs.read(paths=..., columns=...)`. Never a private bypass; never N calls.

In [46]:
# Wrap fs.read with a spy so we can see exactly when and how hydration fires
original_read = db.read
calls: list[dict] = []


async def read_spy(**kwargs):
    calls.append(
        {
            "columns": kwargs.get("columns"),
            "n_paths": len(kwargs.get("candidates").entries) if kwargs.get("candidates") else None,
        }
    )
    return await original_read(**kwargs)


db.read = read_spy  # type: ignore[assignment]

In [47]:
# Ask for `updated_at` via --output. glob already fetches it by default here,
# so hydration is a no-op — spy records no calls.
calls.clear()
_ = await execute_query(db, parse_query('glob "**/*.md" --output default,updated_at'))
print("hydration calls:", calls)

hydration calls: []


In [48]:
# Now simulate a pipeline where upstream hands downstream only paths — the
# executor sees that `updated_at` is null-for-all and backfills via one read.
from vfs.query.ast import QueryPlan, GlobCommand
from vfs.results import VFSResult, Entry
from unittest.mock import AsyncMock


# Replace db.glob with a stub that returns path-only entries (no updated_at)
async def stub_glob(**_):
    return VFSResult(
        function="glob",
        entries=[Entry(path="/docs/intro.md"), Entry(path="/docs/guide.md")],
    )


db_glob_original = db.glob
db.glob = stub_glob  # type: ignore[assignment]
calls.clear()

plan = QueryPlan(
    ast=GlobCommand(pattern="**/*.md"),
    methods=("glob",),
    projection=("path", "updated_at"),
)
result = await execute_query(db, plan)

print("hydration calls:", calls)
print("entries now carry updated_at:")
for e in result.entries:
    print(" ", e.path, "→", e.updated_at)

# Restore real glob
db.glob = db_glob_original  # type: ignore[assignment]

hydration calls: [{'columns': frozenset({'updated_at'}), 'n_paths': 2}]
entries now carry updated_at:
  /docs/intro.md → 2026-04-19 20:35:45.267720
  /docs/guide.md → 2026-04-19 20:35:45.270778


## 7. Persisted graph degrees

`VFSObject` gained indexed `in_degree` / `out_degree` columns so glob / stat / ls can carry them without a graph round-trip. Per Article 7 (no migration scripts), rows stay `NULL` until a graph-refresh pass writes the computed values — the schema is ready but the write path doesn't precompute degrees. `pagerank` / centrality ops override whatever's on the row with freshly-computed values.

In [49]:
# auth.py has one outgoing connection to db.py
stat_auth = await db.stat("/src/auth.py")
stat_db = await db.stat("/src/db.py")
print("auth.py → in:", stat_auth.entries[0].in_degree, "out:", stat_auth.entries[0].out_degree)
print("db.py   → in:", stat_db.entries[0].in_degree, "out:", stat_db.entries[0].out_degree)

auth.py → in: None out: None
db.py   → in: None out: None


In [50]:
# Pagerank — ranks the small graph, overrides in/out_degree with the computed values
pr = await db.pagerank()
print(pr.to_str(projection=("path", "score", "in_degree", "out_degree")))

| path         |  score | in_degree | out_degree |
| ------------ | -----: | --------: | ---------: |
| /src/auth.py | 0.3509 |         0 |          1 |
| /src/db.py   | 0.6491 |         1 |          0 |


## 8. Review repros

These are narrow, runnable examples for the review findings raised on this branch.

In [ ]:
# Fix check: overlapping grep context windows now render as one merged block.
from vfs.results import Entry, LineMatch, VFSResult

overlap = VFSResult(
    function="grep",
    entries=[
        Entry(
            path="/a.py",
            content="one\nhit A\nhit B\nfour\n",
            lines=[
                LineMatch(start=1, end=4, match=2),
                LineMatch(start=1, end=4, match=3),
            ],
        )
    ],
)

rendered = overlap.to_str()
print(rendered)
assert rendered == "/a.py-1-one\n/a.py:2:hit A\n/a.py:3:hit B\n/a.py-4-four"

In [ ]:
# Fix check: tree keeps ASCII tree formatting even when a visibility flag is used.
plain_tree = await db.cli("tree /")
all_tree = await db.cli("tree / --all")

print("tree /")
print(plain_tree)
print("\ntree / --all")
print(all_tree)

assert "└──" in plain_tree or "├──" in plain_tree
assert "└──" in all_tree or "├──" in all_tree

In [55]:
# Fix check: sort no longer accepts --by. Plain sort still works; --by is rejected
# at parse time instead of being silently ignored.
from unittest.mock import AsyncMock
from vfs.query.executor import execute_query
from vfs.query.parser import QuerySyntaxError, parse_query
from vfs.results import Entry, VFSResult

sort_fs = AsyncMock()
sort_fs._merge_results = lambda results: results[0]
sort_fs.glob.return_value = VFSResult(
    function="glob",
    entries=[Entry(path="/a.py", score=0.2), Entry(path="/b.py", score=0.9)],
)

result = await execute_query(sort_fs, parse_query('glob "*.py" | sort'))
print("plain sort order:", [e.path for e in result.entries])

try:
    parse_query('glob "*.py" | sort --by pagerank')
except QuerySyntaxError as exc:
    print("sort --by rejected:", exc)

plain sort order: ['/b.py', '/a.py']


## 9. Tear-down

In [ ]:
event.remove(sync_engine, "before_cursor_execute", _on_execute)
await engine.dispose()
print("done")